In [8]:
# REQUIRES
# ======================================================================================================================================
import time
import polars as pl

In [9]:
# CONSTS & GLOBALS VARIABLES
# Meeeh! But it's ok to this practice
# ======================================================================================================================================
# Consts
# --------------------------------------------------------------------------------------------------------------------------------------
PATH_MOVIES  = "../dataset/ml-20m/movies.csv"
PATH_RATINGS = "../dataset/ml-20m/ratings.csv"
# Globals
# --------------------------------------------------------------------------------------------------------------------------------------
df_pl_ea = {}
df_pl_lz = {}

In [10]:
# HELPERS FUNCTIONS
# ======================================================================================================================================

def LineBreak(pHowMany: int = 1):
  for i in range(0, pHowMany):
    print(f"+ ")

def SectionBreak():
  LineBreak(1)
  print(f"+ --------------------------------------------------------------------------")
  LineBreak(1)

In [11]:
# 1. Cargar el dataset movielens superdataset.csv (ratings.csv in PATH_RATINGS)
# ======================================================================================================================================

def PolarsLazyframe():
  tInit = time.time()

  # Creating the lazyframe
  # No read file yet, only inspect the schema
  df_pl_lz['ratings'] = pl.scan_csv(PATH_RATINGS)
  df_pl_lz['movies'] = pl.scan_csv(PATH_MOVIES)

  # No exectue yet, only build the query plan
  queryPlan = (df_pl_lz['ratings']
    .group_by("movieId")
    .agg(pl.col("rating").mean().alias("avgRating"))
    .join(df_pl_lz['movies'], on="movieId")
    .filter((pl.col("genres").str.contains("Action")) & (pl.col("avgRating") >= 4))
  )

  # Execute with .collect()
  # Polars optimize the query plan and execute it
  resFinal = queryPlan.collect()
  
  print(f"+ Polars Lazy Time: {time.time() - tInit:.4f}s")
  print(f"+ Movies found: {resFinal.height}")


In [12]:
# Eager load mode with Polars
# ======================================================================================================================================

def PolarsEagerframe():
  tInit = time.time()
  # Load & examine immediately
  df_pl_ea['ratings'] = pl.read_csv(PATH_RATINGS)
  df_pl_ea['movies'] = pl.read_csv(PATH_MOVIES)

  # Exectue immediately
  resPl = (df_pl_ea['ratings']
    .group_by("movieId")
    .agg(pl.col("rating").mean().alias("avgRating"))
    .join(df_pl_ea['movies'], on="movieId")
    .filter((pl.col("genres").str.contains("Action")) & (pl.col("avgRating") >= 4))
  )
  
  print(f"+ Polars result: {resPl.height} Filter Time: {time.time() - tInit:.4f}s")



In [13]:
print(f"+ ==========================================================================")
LineBreak(1)
print(f"+ Analysing Polars Eagerframe...")
print(f"+ --------------------------------------------------------------------------")
PolarsEagerframe()
SectionBreak()
print(f"+ Analysing Polars Lazyframe...")
print(f"+ --------------------------------------------------------------------------")
LineBreak(1)
PolarsLazyframe()
LineBreak(1)
print(f"+ =======================================================================")


+ ==========================================================================
+ 
+ Analysing Polars Eagerframe...
+ --------------------------------------------------------------------------
+ Polars result: 118 Filter Time: 1.8532s
+ 
+ --------------------------------------------------------------------------
+ 
+ Analysing Polars Lazyframe...
+ --------------------------------------------------------------------------
+ 
+ Polars Lazy Time: 0.5013s
+ Movies found: 118
+ 
+ =======================================================================
